# btp-eval-optimised: paired evaluation and offloading experiments

Derived from **btp-baseline+**; the original notebook and source dataset remain unchanged.
Use **GPU T4 x2**, Internet on, and save outputs. Inference uses GPU 0; VRAM is not pooled across the two cards.

The default pilot runs two fixed NH-HAZE development crops with M0 and M1 (four cold-process jobs), followed by one separate M0 profiler job. Images are 512×512 paired center crops. These are internal diagnostic results, not official full-resolution benchmark scores.

Run cells in order. Each evaluation creates a new directory, including on reruns. `RUN_PROFILE` may be disabled to save compute. Expand the matrix only after inspecting the pilot cost and failures.


In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

from IPython.display import FileLink, Image, display

subprocess.run(["nvidia-smi"], check=True)
# Attached private weights dataset; update the version if the dataset changes.
WEIGHTS_DATASET_VERSION = "cubingultimate/btp-paper-weights-2/versions/1"
assert not WEIGHTS_DATASET_VERSION.startswith("EDIT_ME"), (
    "Set the dataset version above first"
)
ROOT = Path("/kaggle/working/BTP")
OUTPUT = Path("/kaggle/working/btp-eval-results")
OUTPUT.mkdir(parents=True, exist_ok=True)
RUN_PROFILE = True

## 1. Restore and verify the uploaded source bundle
Kaggle may keep the ZIP or expand it; both layouts are supported. Existing working copies are checked rather than silently overwritten.

In [ ]:
archives = list(Path("/kaggle/input").rglob("btp-eval-source.bundle"))
if not archives:
    archives = list(Path("/kaggle/input").rglob("btp-eval-source.zip"))
if ROOT.exists():
    shutil.move(str(ROOT), str(ROOT.with_name("BTP-backup-" + str(time.time_ns()))))
if not ROOT.exists():
    if len(archives) == 1:
        with zipfile.ZipFile(archives[0]) as archive:
            for item in archive.namelist():
                destination = (Path("/kaggle/working") / item).resolve()
                if not destination.is_relative_to(Path("/kaggle/working")):
                    raise ValueError("Unsafe archive member")
            archive.extractall("/kaggle/working")
    else:
        manifests = list(Path("/kaggle/input").rglob("source-manifest.json"))
        assert len(manifests) == 1, "Attach exactly one BTP source dataset"
        shutil.copytree(manifests[0].parent, ROOT)
snapshot = json.loads((ROOT / "source-manifest.json").read_text())
# Recover the exact upstream gzip if a dataset upload expanded it to .txt.
vocab_relative = "upstream/Learning-Hazing-to-Dehazing/diffbir/model/open_clip/bpe_simple_vocab_16e6.txt.gz"
vocab = ROOT / vocab_relative
if not vocab.exists() and vocab.with_suffix("").exists():
    import gzip
    import urllib.request

    commit = snapshot["upstream_commit"]
    assert len(commit) == 40 and all(c in "0123456789abcdef" for c in commit)
    url = (
        "https://raw.githubusercontent.com/ruiyi-w/Learning-Hazing-to-Dehazing/"
        + commit
        + "/diffbir/model/open_clip/bpe_simple_vocab_16e6.txt.gz"
    )
    with urllib.request.urlopen(url, timeout=60) as response:
        original_gzip = response.read()
    assert (
        hashlib.sha256(original_gzip).hexdigest() == snapshot["files"][vocab_relative]
    ), "Tokenizer download hash mismatch"
    assert gzip.decompress(original_gzip) == vocab.with_suffix("").read_bytes(), (
        "Expanded tokenizer differs from upstream"
    )
    vocab.write_bytes(original_gzip)
    print("Recovered verified tokenizer gzip:", vocab)

for relative, expected in snapshot["files"].items():
    # Finder metadata is not source and can be omitted during dataset upload.
    if Path(relative).name == ".DS_Store":
        continue
    assert hashlib.sha256((ROOT / relative).read_bytes()).hexdigest() == expected, (
        relative
    )
print("Verified source snapshot:", snapshot["upstream_commit"])

## 2. Create an isolated Python 3.10 CUDA runtime
This downloads Python, CUDA-enabled PyTorch and dependencies. The environment is in `/tmp` so it is not included in saved notebook outputs. It leaves Kaggle's notebook-kernel packages intact.

In [ ]:
uv_binary = shutil.which("uv")
if uv_binary is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
    UV = [sys.executable, "-m", "uv"]
else:
    UV = [uv_binary]
ENV = Path("/tmp/btp-kaggle-env")
PYTHON = ENV / "bin/python"
if not PYTHON.exists():
    subprocess.run(UV + ["venv", "--python", "3.10", str(ENV)], check=True)
subprocess.run(
    UV
    + [
        "pip",
        "install",
        "--python",
        str(PYTHON),
        "torch==2.6.0",
        "torchvision==0.21.0",
        "--index-url",
        "https://download.pytorch.org/whl/cu118",
    ],
    check=True,
)
subprocess.run(
    UV
    + [
        "pip",
        "install",
        "--python",
        str(PYTHON),
        "-r",
        str(ROOT / "configs/kaggle-requirements.txt"),
    ],
    check=True,
)
RUN = UV + ["run", "--no-project", "--python", str(PYTHON), "python"]
ENVIRONMENT = os.environ.copy()
ENVIRONMENT.update(
    PYTHONPATH=str(ROOT / "src"), CUDA_VISIBLE_DEVICES="0", ATTN_MODE="sdp"
)
subprocess.run(
    RUN
    + [
        "-c",
        "import torch; assert torch.cuda.is_available(); print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0)); print('VRAM GB:', torch.cuda.get_device_properties(0).total_memory/1e9)",
    ],
    check=True,
    env=ENVIRONMENT,
)
with (OUTPUT / "runtime-packages.txt").open("w") as file:
    subprocess.run(
        UV + ["pip", "freeze", "--python", str(PYTHON)], check=True, stdout=file
    )

# Tiny CPU check in the existing inference environment; no extra packages.
subprocess.run(
    RUN + [str(ROOT / "scripts/smoke_memory.py")], check=True, env=ENVIRONMENT, cwd=ROOT
)

## 3. Locate and hash the three checkpoints
Weights remain on the read-only input mount. The two supplied Baidu files are byte-identical to our previous copies; this step detects incomplete/wrong uploads.

In [ ]:
catalog = json.loads((ROOT / "configs/checkpoints.json").read_text())
paths = {}
for name, spec in catalog.items():
    candidates = list(Path("/kaggle/input").rglob(name))
    assert len(candidates) == 1, f"Expected exactly one {name}, found {len(candidates)}"
    path = candidates[0]
    assert path.stat().st_size == spec["bytes"], f"Incomplete upload: {name}"
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for block in iter(lambda: file.read(8 * 1024 * 1024), b""):
            digest.update(block)
    assert digest.hexdigest() == spec["sha256"], f"Hash mismatch: {name}"
    paths[name] = path
    print("Verified", name, digest.hexdigest())
# Files can be in different dataset directories; symlink them into a common folder.
WEIGHTS = Path("/tmp/btp-reference-weights")
WEIGHTS.mkdir(exist_ok=True)
for name, path in paths.items():
    target = WEIGHTS / name
    if target.is_symlink():
        assert target.resolve() == path.resolve(), (
            "Restart session after changing datasets"
        )
    elif target.exists():
        raise FileExistsError(target)
    else:
        target.symlink_to(path)
(OUTPUT / "weights-verification.json").write_text(
    json.dumps(
        {
            "dataset_version": WEIGHTS_DATASET_VERSION,
            "files": {
                name: {"path": str(path), **catalog[name]}
                for name, path in paths.items()
            },
        },
        indent=2,
    )
)

## 4. Tiny CPU smoke first
Random tiny networks test plumbing only. Their images are not pretrained results. This cell does not run training or a large model.

In [ ]:
subprocess.run(RUN + ["-m", "btp.model_smoke"], cwd=ROOT, env=ENVIRONMENT, check=True)

## 5. Restore the frozen paired dataset
The manifest verifies every image hash and pair, including held-out scenes. Only the two preselected development scenes enter the pilot. Source hashes, crop coordinates and split rules are available for inspection.


In [ ]:
archives = list(Path("/kaggle/input").rglob("nh-haze-512.bundle"))
assert len(archives) == 1, "Attach cubingultimate/btp-eval-nh-haze"
DATA = Path("/kaggle/working/nh-haze-eval-" + str(time.time_ns()))
DATA.mkdir()
with zipfile.ZipFile(archives[0]) as archive:
    for item in archive.namelist():
        assert (DATA / item).resolve().is_relative_to(DATA.resolve()), (
            "Unsafe dataset path"
        )
    archive.extractall(DATA)
MANIFEST = DATA / "nh_haze_manifest.json"
assert (
    MANIFEST.read_bytes()
    == (ROOT / "configs/evaluation/nh_haze_manifest.json").read_bytes()
), "Evaluation dataset version mismatch"
subprocess.run(
    RUN + ["-m", "btp.evaluation", "--manifest", str(MANIFEST)],
    cwd=ROOT,
    env=ENVIRONMENT,
    check=True,
)
provenance = {
    "notebook": "cubingultimate/btp-eval-optimised",
    "source_dataset": "cubingultimate/btp-eval-source",
    "evaluation_dataset": "cubingultimate/btp-eval-nh-haze/versions/1",
    "evaluation_archive_sha256": hashlib.sha256(archives[0].read_bytes()).hexdigest(),
    "source_manifest_sha256": hashlib.sha256(
        (ROOT / "source-manifest.json").read_bytes()
    ).hexdigest(),
    "manifest_sha256": hashlib.sha256(MANIFEST.read_bytes()).hexdigest(),
    "baseline_template_sha256": "1ef0c9a1bb461698bdbd0459a3655f8715784e84c0a39761f57889330bce8522",
}
(OUTPUT / "notebook-provenance.json").write_text(json.dumps(provenance, indent=2))
with (OUTPUT / "nvidia-smi.txt").open("w") as file:
    subprocess.run(["nvidia-smi"], stdout=file, check=True)
print(json.dumps(provenance, indent=2))

## 6. Run the bounded development pilot
M0 is the existing low-memory placement. M1 skips denoiser offload/restore when the guidance window is inactive. Both keep the same weights, precision, prompts, guidance settings, geometry and seed. Preview decoding is disabled for all measured runs.

Failures are retained in `jobs.jsonl`; quality is computed only for valid outputs and comparisons use matched successful jobs. Each worker has a timeout. `repeatability.json` expands to 12 jobs; `placement.json` expands to 30. Do not switch to those before reviewing pilot runtime and GPU quota.


In [ ]:
def run_evaluation(config_name):
    config_path = ROOT / "configs/evaluation" / (config_name + ".json")
    config = json.loads(config_path.read_text())
    print(json.dumps(config, indent=2))
    destination = OUTPUT / (config_name + "-" + str(time.time_ns()))
    subprocess.run(
        RUN
        + [
            "-u",
            "-m",
            "btp.benchmark",
            "--manifest",
            str(MANIFEST),
            "--config",
            str(config_path),
            "--weights",
            str(WEIGHTS),
            "--output",
            str(destination),
        ],
        cwd=ROOT,
        env=ENVIRONMENT,
        check=True,
    )
    return destination


PILOT = run_evaluation("pilot")
summary = json.loads((PILOT / "summary.json").read_text())
print(json.dumps(summary, indent=2))

## 7. Inspect quality, latency, memory and paired outputs
Timing rows are cold launches; repeated launches do not constitute a resident service. Sampling time synchronizes CUDA at its boundaries. Phase enqueue times overlap and must not be added together. Two scenes and one repeat establish plumbing only; they cannot establish a speedup or equivalence.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image as PILImage

jobs = [json.loads(line) for line in (PILOT / "jobs.jsonl").read_text().splitlines()]
display(pd.read_csv(PILOT / "per_image.csv"))
rows = []
for job in jobs:
    telemetry = job.get("telemetry", {})
    rows.append(
        {
            "scene": job["scene_id"],
            "policy": job["policy"],
            "status": job["status"],
            "sampling_seconds": job.get("sampling_seconds"),
            "lifetime_peak_GiB": telemetry.get("lifetime_peak_allocated_bytes", 0)
            / 2**30,
            "sampling_peak_GiB": telemetry.get("sampling", {}).get(
                "peak_allocated_bytes", 0
            )
            / 2**30,
            "transfer_payload_GiB": sum(
                telemetry.get("transfers", {}).get(k, 0)
                for k in ("to_cpu_bytes", "to_gpu_bytes")
            )
            / 2**30,
        }
    )
display(pd.DataFrame(rows))
good = [job for job in jobs if job["status"] == "completed"]
if good:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    labels = [job["scene_id"] + "/" + job["policy"] for job in good]
    axes[0].bar(labels, [job["sampling_seconds"] for job in good])
    axes[0].set_ylabel("Sampling seconds")
    axes[1].bar(labels, [job["metrics"]["psnr_db"] or 0 for job in good])
    axes[1].set_ylabel("PSNR dB (exact matches shown as 0)")
    for ax in axes:
        ax.tick_params(axis="x", rotation=45)
    fig.tight_layout()
    fig.savefig(PILOT / "pilot-comparison.png", dpi=150)
    plt.show()
    for job in good:
        strip = PILImage.new("RGB", (1536, 512))
        for index, key in enumerate(("input", "prediction", "target")):
            with PILImage.open(job[key]) as img:
                strip.paste(img, (index * 512, 0))
        path = PILOT / (job["job_id"] + "-hazy-prediction-clean.png")
        strip.save(path)
        print(job["scene_id"], job["policy"], "hazy | prediction | clean")
        display(Image(filename=str(path), width=1000))

## 8. Separate diagnostic profile
This extra one-scene M0 job writes a Chrome/Perfetto trace. It is excluded from the unprofiled pilot table. Inspect offload/restore, VAE decode, backward, denoiser and allocator ranges before choosing a placement policy. Phase counters and tensor payload estimates are not measured PCIe traffic.


In [ ]:
PROFILE = None
if RUN_PROFILE and any(
    job["policy"] == "M0" and job["status"] == "completed" for job in jobs
):
    PROFILE = run_evaluation("profile")
    print("Profiler artifacts:", list(PROFILE.rglob("profile.json")))
else:
    print("Profile skipped: disabled or no successful M0 pilot.")

## 9. Save evidence
The archive includes job logs, generated configurations, checkpoint hashes, predictions, per-image metrics, summaries, memory counters, runtime package versions and the optional diagnostic trace. Failed jobs remain visible.

LPIPS is optional and disabled in this first pilot; it requires a separate explicit scoring dependency/weight download. M4 resident execution and asynchronous transfers are research follow-ups. Unsupported execution modes are rejected rather than mislabeled as completed optimizations.


In [ ]:
archive = shutil.make_archive("/kaggle/working/btp-eval-results", "zip", OUTPUT)
display(FileLink(archive))
failures = [job for job in jobs if job["status"] != "completed"]
print(
    "Pilot:",
    len(jobs) - len(failures),
    "/",
    len(jobs),
    "completed; failures:",
    [(j["job_id"], j["status"]) for j in failures],
)
print("Saved:", archive)